In [ ]:
# ============================================================
# 01_eda.ipynb  —  Exploratory Data Analysis
# Run as a plain .py or paste cells into Jupyter
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted")
DATA_PATH = "../data/processed/features.csv"

# --- 1. Load & Overview ---
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print(df.dtypes)
print(df.describe())
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nLabel distribution:\n{df['phq_label'].value_counts()}")

# --- 2. Label Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["phq_label"].value_counts().plot(
    kind="bar", ax=axes[0], color=["#68d391", "#fc8181"]
)
axes[0].set_title("Depression Label Distribution")
axes[0].set_xticklabels(["Not Depressed (0)", "Depressed (1)"], rotation=0)

df["phq_score"].hist(bins=27, ax=axes[1], color="#4c51bf", alpha=0.7)
axes[1].axvline(10, color="red", linestyle="--", label="PHQ-9 ≥ 10 = Depressed")
axes[1].set_title("PHQ-9 Score Distribution")
axes[1].legend()
plt.tight_layout()
plt.savefig("../reports/label_distribution.png", dpi=150)
plt.show()

# --- 3. Feature Correlation Heatmap ---
FEATURE_COLS = [
    "radius_of_gyration", "home_stay_pct", "location_entropy",
    "num_places_visited", "daily_step_count", "sedentary_bout_count",
    "sedentary_total_mins", "circadian_index", "interdaily_stability",
    "intradaily_variability", "sleep_duration_hrs", "sleep_midpoint_hr",
    "sleep_disruption_count", "call_frequency", "call_duration_avg",
    "screen_unlock_count", "screen_on_duration_mins"
]

corr_matrix = df[FEATURE_COLS + ["phq_label"]].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, square=True, cbar_kws={"shrink": 0.8}
)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("../reports/correlation_heatmap.png", dpi=150)
plt.show()

# --- 4. Feature Distributions by Label ---
fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLS[:16]):
    sns.boxplot(
        x="phq_label", y=col, data=df,
        palette={0: "#68d391", 1: "#fc8181"},
        ax=axes[i]
    )
    axes[i].set_title(col.replace("_", " ").title(), fontsize=10)
    axes[i].set_xlabel("")

plt.suptitle("Feature Distributions by Depression Label", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("../reports/feature_distributions.png", dpi=150)
plt.show()

# --- 5. Statistical Significance Tests ---
print("\n=== Mann-Whitney U Test (Depressed vs Not) ===")
for col in FEATURE_COLS:
    dep   = df[df["phq_label"] == 1][col].dropna()
    nodep = df[df["phq_label"] == 0][col].dropna()
    stat, p = stats.mannwhitneyu(dep, nodep, alternative="two-sided")
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))
    print(f"{col:<35} p={p:.4f}  {sig}")

# --- 6. Correlation with PHQ-9 score (continuous) ---
print("\n=== Pearson Correlation with PHQ-9 Score ===")
corr_with_phq = df[FEATURE_COLS + ["phq_score"]].corr()["phq_score"].drop("phq_score")
print(corr_with_phq.sort_values(key=abs, ascending=False).to_string())

# --- 7. Missing data imputation check ---
print(f"\nRows with any missing feature: {df[FEATURE_COLS].isnull().any(axis=1).sum()}")
df_clean = df.dropna(subset=FEATURE_COLS)
print(f"Clean rows after dropna: {len(df_clean)}")